In [4]:
from google.colab import drive

drive.mount('/content/gdrive')

Mounted at /content/gdrive


In [5]:
from pathlib import Path

drive_root = Path("/content/gdrive/MyDrive")

matches = list(
    drive_root.rglob("synthetic_gps_data.csv")
)

print("CSV files found:", len(matches))

for file_path in matches:
    print(file_path)

CSV files found: 1
/content/gdrive/MyDrive/GeoPulse-GitHub-Repo/data/synthetic/synthetic_gps_data.csv


In [6]:
import pandas as pd

DATA_PATH = "/content/gdrive/MyDrive/GeoPulse-GitHub-Repo/data/synthetic/synthetic_gps_data.csv"

gps_df = pd.read_csv(DATA_PATH)

gps_df["timestamp"] = pd.to_datetime(gps_df["timestamp"])

print("GeoPulse - Day 03")
print("Dataset loaded successfully!")
print("Total GPS records:", len(gps_df))
print("Unique devices:", gps_df["DeviceID"].nunique())

display(gps_df.head())

GeoPulse - Day 03
Dataset loaded successfully!
Total GPS records: 50000
Unique devices: 1000


,DeviceID,latitude,longitude,timestamp
0,DEVICE_00001,18.552963,73.784873,2026-11-17 01:32:09
1,DEVICE_00001,18.642396,73.943783,2026-11-17 04:22:55
2,DEVICE_00001,18.401381,73.953865,2026-11-17 10:18:30
3,DEVICE_00001,18.593061,73.799679,2026-11-17 12:12:06
4,DEVICE_00001,18.497169,73.817837,2026-11-17 13:35:23


In [7]:
# GeoPulse - Day 03
# Define synthetic urban activity zones in Pune

activity_zones = {
    "residential": {
        "latitude": 18.50,
        "longitude": 73.85
    },
    "office": {
        "latitude": 18.56,
        "longitude": 73.91
    },
    "shopping": {
        "latitude": 18.52,
        "longitude": 73.88
    },
    "food": {
        "latitude": 18.53,
        "longitude": 73.87
    },
    "leisure": {
        "latitude": 18.51,
        "longitude": 73.86
    }
}

print("Synthetic Pune activity zones created:\n")

for zone, location in activity_zones.items():
    print(
        zone.title(),
        "-> Latitude:", location["latitude"],
        "| Longitude:", location["longitude"]
    )

Synthetic Pune activity zones created:

Residential -> Latitude: 18.5 | Longitude: 73.85
Office -> Latitude: 18.56 | Longitude: 73.91
Shopping -> Latitude: 18.52 | Longitude: 73.88
Food -> Latitude: 18.53 | Longitude: 73.87
Leisure -> Latitude: 18.51 | Longitude: 73.86


In [8]:
# Define simulated daily urban movement patterns

daily_patterns = {
    "morning": {
        "hours": (6, 10),
        "zones": ["residential", "office"]
    },
    "midday": {
        "hours": (10, 16),
        "zones": ["office", "food", "shopping"]
    },
    "evening": {
        "hours": (16, 20),
        "zones": ["office", "shopping", "residential"]
    },
    "night": {
        "hours": (20, 24),
        "zones": ["leisure", "residential"]
    }
}

print("Daily urban movement patterns defined:\n")

for period, details in daily_patterns.items():
    print(
        period.title(),
        "| Hours:", details["hours"],
        "| Activity zones:", details["zones"]
    )

Daily urban movement patterns defined:

Morning | Hours: (6, 10) | Activity zones: ['residential', 'office']
Midday | Hours: (10, 16) | Activity zones: ['office', 'food', 'shopping']
Evening | Hours: (16, 20) | Activity zones: ['office', 'shopping', 'residential']
Night | Hours: (20, 24) | Activity zones: ['leisure', 'residential']


In [9]:
# Generate synthetic GPS coordinates around an activity zone

def generate_location(zone_name, spread=0.005):
    center = activity_zones[zone_name]

    latitude = np.random.normal(
        center["latitude"], spread
    )

    longitude = np.random.normal(
        center["longitude"], spread
    )

    return latitude, longitude


# Test the location generator
for zone in activity_zones:
    lat, lon = generate_location(zone)

    print(
        f"{zone.title():12s} -> "
        f"Latitude: {lat:.6f}, "
        f"Longitude: {lon:.6f}"
    )

Residential  -> Latitude: 18.502484, Longitude: 73.849309
Office       -> Latitude: 18.563238, Longitude: 73.917615
Shopping     -> Latitude: 18.518829, Longitude: 73.878829
Food         -> Latitude: 18.537896, Longitude: 73.873837
Leisure      -> Latitude: 18.507653, Longitude: 73.862713


In [10]:
# Generate timestamps based on daily activity periods

def generate_activity_hour(period):
    start_hour, end_hour = daily_patterns[period]["hours"]

    hour = np.random.randint(start_hour, end_hour)
    minute = np.random.randint(0, 60)
    second = np.random.randint(0, 60)

    return hour, minute, second


# Test timestamp generation
for period in daily_patterns:
    hour, minute, second = generate_activity_hour(period)

    print(
        f"{period.title():10s} -> "
        f"{hour:02d}:{minute:02d}:{second:02d}"
    )

Morning    -> 09:59:20
Midday     -> 10:11:57
Evening    -> 17:43:24
Night      -> 20:26:58


In [11]:
# Generate one synthetic mobility record

def generate_mobility_record(device_id, date, period):
    zone_name = random.choice(
        daily_patterns[period]["zones"]
    )

    latitude, longitude = generate_location(zone_name)

    hour, minute, second = generate_activity_hour(period)

    timestamp = pd.Timestamp(date).replace(
        hour=hour,
        minute=minute,
        second=second
    )

    return {
        "DeviceID": device_id,
        "latitude": latitude,
        "longitude": longitude,
        "timestamp": timestamp,
        "activity_period": period,
        "activity_zone": zone_name
    }


# Test with one device
sample_record = generate_mobility_record(
    "DEVICE_00001",
    "2026-11-18",
    "morning"
)

print("Sample synthetic mobility record:")
display(pd.DataFrame([sample_record]))

Sample synthetic mobility record:


,DeviceID,latitude,longitude,timestamp,activity_period,activity_zone
0,DEVICE_00001,18.497341,73.849413,2026-11-18 09:15:14,morning,residential


In [12]:
# Generate synthetic urban mobility data

START_DATE = pd.Timestamp("2026-11-18")
NUM_DAYS = 7
PINGS_PER_DEVICE_PER_DAY = 20

mobility_records = []

for device_id in gps_df["DeviceID"].unique():

    for day in range(NUM_DAYS):

        current_date = START_DATE + pd.Timedelta(days=day)

        for _ in range(PINGS_PER_DEVICE_PER_DAY):

            period = random.choices(
                ["morning", "midday", "evening", "night"],
                weights=[25, 30, 25, 20]
            )[0]

            record = generate_mobility_record(
                device_id,
                current_date,
                period
            )

            mobility_records.append(record)

mobility_df = pd.DataFrame(mobility_records)

mobility_df = mobility_df.sort_values(
    ["DeviceID", "timestamp"]
).reset_index(drop=True)

print("Synthetic urban mobility dataset generated!")
print("Total records:", len(mobility_df))
print("Unique devices:", mobility_df["DeviceID"].nunique())

display(mobility_df.head(10))

Synthetic urban mobility dataset generated!
Total records: 140000
Unique devices: 1000


,DeviceID,latitude,longitude,timestamp,activity_period,activity_zone
0,DEVICE_00001,18.499878,73.851778,2026-11-18 06:50:06,morning,residential
1,DEVICE_00001,18.501044,73.840202,2026-11-18 07:03:53,morning,residential
2,DEVICE_00001,18.560459,73.900062,2026-11-18 07:36:22,morning,office
3,DEVICE_00001,18.558068,73.904206,2026-11-18 07:39:03,morning,office
4,DEVICE_00001,18.564865,73.913978,2026-11-18 07:54:13,morning,office
5,DEVICE_00001,18.562289,73.901579,2026-11-18 08:36:46,morning,office
6,DEVICE_00001,18.567328,73.908871,2026-11-18 08:43:54,morning,office
7,DEVICE_00001,18.489108,73.844781,2026-11-18 09:13:30,morning,residential
8,DEVICE_00001,18.531620,73.868075,2026-11-18 10:24:06,midday,food
9,DEVICE_00001,18.559159,73.915824,2026-11-18 10:47:54,midday,office


In [13]:
# GeoPulse - Day 03
# Validate synthetic urban mobility data

print("DATASET VALIDATION")
print("-" * 40)

print("Total records:", len(mobility_df))
print("Unique devices:", mobility_df["DeviceID"].nunique())

print("\nMissing values:")
print(mobility_df.isnull().sum())

print("\nActivity period distribution:")
print(mobility_df["activity_period"].value_counts())

print("\nActivity zone distribution:")
print(mobility_df["activity_zone"].value_counts())

print("\nTimestamp range:")
print("Start:", mobility_df["timestamp"].min())
print("End:", mobility_df["timestamp"].max())

print("\nRecords per device:")
print(mobility_df.groupby("DeviceID").size().describe())

print("\nLatitude valid:",
      mobility_df["latitude"].between(-90, 90).all())

print("Longitude valid:",
      mobility_df["longitude"].between(-180, 180).all())

print("Missing values check:",
      not mobility_df.isnull().any().any())

DATASET VALIDATION
----------------------------------------
Total records: 140000
Unique devices: 1000

Missing values:
DeviceID           0
latitude           0
longitude          0
timestamp          0
activity_period    0
activity_zone      0
dtype: int64

Activity period distribution:
activity_period
midday     42025
morning    35316
evening    34930
night      27729
Name: count, dtype: int64

Activity zone distribution:
activity_zone
residential    43265
office         42945
shopping       25750
food           14138
leisure        13902
Name: count, dtype: int64

Timestamp range:
Start: 2026-11-18 06:00:00
End: 2026-11-24 23:59:58

Records per device:
count    1000.0
mean      140.0
std         0.0
min       140.0
25%       140.0
50%       140.0
75%       140.0
max       140.0
dtype: float64

Latitude valid: True
Longitude valid: True
Missing values check: True


In [14]:
from pathlib import Path

OUTPUT_PATH = Path(
    "/content/gdrive/MyDrive/GeoPulse-GitHub-Repo/data/synthetic/realistic_urban_mobility.csv"
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

mobility_df.to_csv(OUTPUT_PATH, index=False)

print("Dataset saved successfully!")
print("File:", OUTPUT_PATH)
print("Records:", len(mobility_df))
print("File size (MB):", round(OUTPUT_PATH.stat().st_size / (1024 * 1024), 2))

Dataset saved successfully!
File: /content/gdrive/MyDrive/GeoPulse-GitHub-Repo/data/synthetic/realistic_urban_mobility.csv
Records: 140000
File size (MB): 11.42
